In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It embeds the required Gaussian output file (water.log)
# directly in this notebook so no external downloads are needed.
#
# No additional package installations are required — this notebook uses only
# Python standard library modules (math, os, etc.).

import sys, os, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# Embedded content of water.log (B3LYP/6-31G* optimization + frequency of H2O)
_WATER_LOG_CONTENT = """
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

# Write to disk in the current working directory
if not os.path.exists("water.log"):
    with open("water.log", "w") as _f:
        _f.write(_WATER_LOG_CONTENT)
    print("Written : water.log")
else:
    print("Found existing: water.log")

print("Environment ready ✓")


# Python for Chemists — Part 2: Parsing Gaussian 16 Output Files
## Demonstration Walkthrough

This notebook is a **guided demonstration** of how to parse and analyse Gaussian 16 output files using Python. Each section introduces a concept, shows the working code, and explains the thought process behind it. Run the cells top to bottom to follow along.

**What this notebook covers**

| Section | Topic |
|---------|-------|
| 1 | Anatomy of a Gaussian `.log` file |
| 2 | Extracting SCF energies |
| 3 | Extracting thermochemical data |
| 4 | Verifying normal termination |
| 5 | Parsing the Standard orientation geometry |
| 6 | Computing bond lengths |
| 7 | Computing bond angles |
| 8 | A complete, reusable parser |

---
## Section 1 — Anatomy of a Gaussian Output File

Gaussian writes a single plain-text `.log` (or `.out`) file recording everything that
happened during a calculation. The sections appear in this order for a geometry
optimisation followed by a frequency calculation:

```
Job header  (version, machine, route card)
Molecule specification  (charge, multiplicity, Z-matrix or Cartesian input)
━━━ Optimisation loop — repeats until convergence ━━━━━━━━━━━━━━━━━━━━━━━━━━
  Input orientation       ← atomic coordinates at this step
  SCF Done                ← electronic energy (Hartree) at this geometry
  Converged? → if not, compute gradient → take new step
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Standard orientation      ← final optimised coordinates in principal-axis frame
Rotational constants
━━━ Thermochemistry ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Zero-point correction
  Thermal corrections (Energy, Enthalpy, Gibbs)
  Sum of electronic + ZPE / H / G
  Harmonic vibrational frequencies
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Normal termination        ← ALWAYS verify this is present before using results
```

The cell below writes a realistic Gaussian output file for a water molecule
(B3LYP/6-31G* geometry optimisation + frequency calculation) that we will parse
throughout this notebook. **Run it now** so the file exists on disk.


In [ ]:
gaussian_text = r"""
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

with open("water.log", "w") as fh:
    fh.write(gaussian_text)

with open("water.log", "r") as fh:
    all_lines = fh.readlines()

print(f"water.log written — {len(all_lines)} lines total")
print()
print("First 8 lines:")
for line in all_lines[:8]:
    print(repr(line))


Notice two things about how Python reads this file:

1. Each line is a **string** ending in `\n` — use `.strip()` to remove it.
2. Some lines have lots of leading whitespace — `.strip()` removes that too.

The string methods you practised in Part 1 (`strip`, `split`, `startswith`, `in`)
are the main tools for navigating this file.


---
## Section 2 — Extracting SCF Energies

The **SCF Done** line is printed once per optimisation step.
Each one looks like:

```
 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
```

After splitting on whitespace the tokens are:
```
index:  0         1          2   3   4                5       6  7  8
token: 'SCF'   'Done:'  'E(RB3LYP)' '=' '-76.4187385461' 'A.U.' ...
```

The energy is **always at index 4**.
For a geometry optimisation you want the **last** `SCF Done` line — the converged energy.


In [ ]:
H2K = 627.509   # Hartree → kcal/mol

scf_energies = []
with open("water.log", "r") as fh:
    for line in fh:
        if "SCF Done:" in line:
            energy = float(line.split()[4])
            scf_energies.append(energy)

print(f"SCF energies found: {len(scf_energies)}")
for i, e in enumerate(scf_energies, start=1):
    print(f"  Step {i}: {e:.10f} Hartree")

print()
print(f"Final (converged) energy : {scf_energies[-1]:.10f} Hartree")
print(f"Energy lowering (1→last) : {(scf_energies[-1] - scf_energies[0]) * H2K:.4f} kcal/mol")


The loop above scans every line for the `SCF Done` pattern and converts the extracted energy string to a float. The final value in `scf_energies` corresponds to the last SCF cycle — typically the converged geometry. Converting to kcal mol⁻¹ with the Hartree conversion factor (627.5094 kcal mol⁻¹ H⁻¹) lets you compare directly with experimental values or other levels of theory.

---
## Section 3 — Thermochemical Corrections

After a frequency calculation Gaussian prints a thermochemistry block.
Each quantity appears on a line that **starts with a recognisable string**,
and the numerical value is always the **last token** on the line.

```
 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128
```

The strategy: keep a dictionary mapping each search string to a short key name,
then scan the file once and populate the dictionary.


In [ ]:
THERMO_KEYS = {
    "Zero-point correction=":                       "zpe_correction",
    "Thermal correction to Enthalpy=":              "thermal_H",
    "Thermal correction to Gibbs Free Energy=":     "thermal_G",
    "Sum of electronic and zero-point Energies=":   "E_zpe",
    "Sum of electronic and thermal Enthalpies=":    "H_total",
    "Sum of electronic and thermal Free Energies=": "G_total",
}

thermo = {}
with open("water.log", "r") as fh:
    for line in fh:
        stripped = line.strip()
        for search, key in THERMO_KEYS.items():
            if stripped.startswith(search):
                thermo[key] = float(stripped.split()[-1])

H2K = 627.509
print(f"{'Quantity':<28} {'Hartree':>14}  {'kcal/mol':>12}")
print("-" * 58)
for key, val in thermo.items():
    print(f"{key:<28} {val:>14.6f}  {val * H2K:>12.3f}")


Gaussian's thermochemistry block appears once, near the bottom of the log file. The keywords `Zero-point correction`, `Thermal correction to Enthalpy`, and `Thermal correction to Gibbs Free Energy` each appear on their own line followed by the value in Hartrees. Storing them in a dictionary makes downstream arithmetic clean and self-documenting.

---
## Section 4 — Verifying Normal Termination

> **Rule:** Never use results from a job that did not terminate normally.

If Gaussian crashes, runs out of time, or hits a disk quota, the last line will
be something like `Error termination` instead of `Normal termination`.
Always check before reporting numbers.


In [ ]:
def check_termination(filename):
    """
    Return (terminated_normally, timestamp) for a Gaussian log file.
    timestamp is a string like 'Mon Mar 14 12:34:56 2026', or None.
    """
    with open(filename, "r") as fh:
        lines = fh.readlines()

    for line in lines:
        if "Normal termination" in line:
            tokens = line.split()
            idx = tokens.index("at")
            timestamp = " ".join(tokens[idx + 1:]).rstrip(".")
            return True, timestamp

    return False, None

ok, ts = check_termination("water.log")
if ok:
    print(f"✓ Job completed normally ({ts})")
else:
    print("✗ WARNING: job did NOT terminate normally — do not use these results!")


Checking for `Normal termination` before using any results is a non-negotiable habit in computational chemistry. Gaussian writes this string as the very last line of a successful job. Its absence means the job crashed, timed out, or the SCF failed to converge — in any of these cases the energies and geometries in the file may be meaningless.

---
## Section 5 — Parsing the Standard Orientation

The **Standard orientation** block contains the final atomic coordinates in Ångströms,
in the molecule's principal-axis frame. It appears after the last optimisation step:

```
                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------
```

**Parsing strategy:** use a boolean flag that turns on when you see
`"Standard orientation:"` and turns off when you hit the closing dashes.
Reset the atoms list each time you enter a new block so you always keep the **last** one
(relevant for multi-step jobs).

Column layout of each coordinate line (after splitting on whitespace):

| Index | Content |
|-------|---------|
| 0 | Center number |
| 1 | **Atomic number** |
| 2 | Atomic type |
| 3 | **X** |
| 4 | **Y** |
| 5 | **Z** |


In [ ]:
ATOMIC_SYMBOLS = {
    1: "H",  5: "B",  6: "C",  7: "N",  8: "O",  9: "F",
   14: "Si", 15: "P", 16: "S", 17: "Cl", 35: "Br", 53: "I",
}

def parse_standard_orientation(filename):
    """
    Return a list of atom dicts from the last Standard orientation block.
    Each dict: {'symbol': str, 'x': float, 'y': float, 'z': float}
    """
    atoms        = []
    in_block     = False
    header_skip  = 0

    with open(filename, "r") as fh:
        for line in fh:
            if "Standard orientation:" in line:
                in_block, header_skip, atoms = True, 4, []
                continue
            if in_block:
                if header_skip > 0:
                    header_skip -= 1
                    continue
                if line.strip().startswith("-----"):
                    in_block = False
                    continue
                t = line.split()
                sym = ATOMIC_SYMBOLS.get(int(t[1]), f"Z{t[1]}")
                atoms.append({"symbol": sym,
                              "x": float(t[3]), "y": float(t[4]), "z": float(t[5])})
    return atoms

atoms = parse_standard_orientation("water.log")

print(f"{'#':<4} {'Sym':<6} {'X (Å)':>10} {'Y (Å)':>10} {'Z (Å)':>10}")
print("-" * 44)
for i, a in enumerate(atoms, 1):
    print(f"{i:<4} {a['symbol']:<6} {a['x']:>10.6f} {a['y']:>10.6f} {a['z']:>10.6f}")


The Standard orientation block lists every atom in Cartesian coordinates after the optimiser has rotated the molecule into a canonical frame. Parsing it gives us the full 3-D geometry as a list of `(symbol, x, y, z)` tuples — the foundation for every subsequent geometric analysis.

---
## Section 6 — Computing Bond Lengths

Given two atoms at positions $(x_1, y_1, z_1)$ and $(x_2, y_2, z_2)$,
the Euclidean distance is:

$$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2 + (z_2-z_1)^2}$$

This is implemented with Python's `math.sqrt` from the standard library,
which you import once at the top of the cell.


In [ ]:
import math

def bond_length(atom1, atom2):
    """Euclidean distance (Å) between two atom dicts with keys x, y, z."""
    dx = atom2["x"] - atom1["x"]
    dy = atom2["y"] - atom1["y"]
    dz = atom2["z"] - atom1["z"]
    return math.sqrt(dx**2 + dy**2 + dz**2)

# atoms[0]=O, atoms[1]=H1, atoms[2]=H2
r_OH1 = bond_length(atoms[0], atoms[1])
r_OH2 = bond_length(atoms[0], atoms[2])
r_HH  = bond_length(atoms[1], atoms[2])

print(f"O–H1 : {r_OH1:.4f} Å")
print(f"O–H2 : {r_OH2:.4f} Å")
print(f"H–H  : {r_HH:.4f} Å")
print(f"Mean O–H : {(r_OH1 + r_OH2) / 2:.4f} Å  (experiment: 0.9572 Å)")


Bond lengths are the most basic geometric property. The formula above is just the Euclidean distance between two points in three-dimensional space, which `math.dist` computes in a single call. A typical C–H bond is ~1.09 Å, C–C ~1.54 Å, and C=O ~1.23 Å — values you can use as sanity checks on a new geometry.

---
## Section 7 — Computing Bond Angles

The bond angle at atom **B** in the arrangement **A–B–C** is found using the dot product
of vectors $\vec{BA}$ and $\vec{BC}$:

$$\cos\theta = \frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}
\qquad
\theta = \arccos\!\left(\frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}\right)$$

Note the `max(-1, min(1, ...))` clamp in the example — floating-point rounding can
occasionally push the cosine just outside $[-1, 1]$, causing `math.acos` to raise a
`ValueError`.


In [ ]:
def bond_angle(atomA, atomB, atomC):
    """
    A–B–C angle in degrees. B is the central atom.
    Uses the dot-product formula with a clamp to avoid acos domain errors.
    """
    BA = [atomA[k] - atomB[k] for k in ("x", "y", "z")]
    BC = [atomC[k] - atomB[k] for k in ("x", "y", "z")]

    dot     = sum(BA[i] * BC[i] for i in range(3))
    mag_BA  = math.sqrt(sum(v**2 for v in BA))
    mag_BC  = math.sqrt(sum(v**2 for v in BC))
    cos_ang = max(-1.0, min(1.0, dot / (mag_BA * mag_BC)))
    return math.degrees(math.acos(cos_ang))

# H1–O–H2  (O is the central atom, index 0)
angle_HOH = bond_angle(atoms[1], atoms[0], atoms[2])
print(f"H–O–H angle:  {angle_HOH:.4f}°")
print(f"Experimental: 104.45°")
print(f"Difference:   {abs(angle_HOH - 104.45):.4f}°")


The dot-product formula for the bond angle converts naturally to Python using `numpy.dot` and `numpy.arccos`. The key step is forming the two bond vectors by subtracting the central atom's coordinates from each terminal atom's coordinates, then normalising before taking the dot product so the result falls in [−1, 1] and `arccos` is well-defined. Converting the result from radians to degrees with `math.degrees` gives values you can compare directly with crystallographic data or textbook values.

---
## Section 8 — A Complete, Reusable Parser

Now we combine everything into a single function. Good design means:

- **One pass** through the file (open once, collect everything)
- A **clean return value** — a dictionary so callers can access results by name
- **Defensive defaults** — set everything to `None` or `[]` before the loop,
  so partial files don't crash the caller

Study the structure carefully: the same flag-based pattern for the Standard orientation
block and the same `startswith` pattern for thermochemistry — just combined.


In [ ]:
import math

ATOMIC_SYMBOLS = {
    1: "H",  5: "B",  6: "C",  7: "N",  8: "O",  9: "F",
   14: "Si", 15: "P", 16: "S", 17: "Cl", 35: "Br", 53: "I",
}
THERMO_KEYS = {
    "Zero-point correction=":                       "zpe_correction",
    "Thermal correction to Enthalpy=":              "thermal_H",
    "Thermal correction to Gibbs Free Energy=":     "thermal_G",
    "Sum of electronic and zero-point Energies=":   "E_zpe",
    "Sum of electronic and thermal Enthalpies=":    "H_total",
    "Sum of electronic and thermal Free Energies=": "G_total",
}

def parse_gaussian(filename):
    """
    Parse a Gaussian log file in one pass.

    Returns
    -------
    dict with keys:
      normal_termination  (bool)
      timestamp           (str or None)
      scf_energies        (list of float)
      final_scf_energy    (float or None)
      thermo              (dict)
      atoms               (list of dicts with symbol, x, y, z)
    """
    result = {
        "normal_termination": False,
        "timestamp":          None,
        "scf_energies":       [],
        "final_scf_energy":   None,
        "thermo":             {},
        "atoms":              [],
    }
    in_std = False
    hskip  = 0

    with open(filename, "r") as fh:
        for line in fh:
            # 1. Normal termination
            if "Normal termination" in line:
                result["normal_termination"] = True
                tokens = line.split()
                idx = tokens.index("at")
                result["timestamp"] = " ".join(tokens[idx+1:]).rstrip(".")

            # 2. SCF energy
            if "SCF Done:" in line:
                result["scf_energies"].append(float(line.split()[4]))

            # 3. Thermochemistry
            stripped = line.strip()
            for search, key in THERMO_KEYS.items():
                if stripped.startswith(search):
                    result["thermo"][key] = float(stripped.split()[-1])

            # 4. Standard orientation
            if "Standard orientation:" in line:
                in_std, hskip, result["atoms"] = True, 4, []
                continue
            if in_std:
                if hskip > 0:
                    hskip -= 1; continue
                if line.strip().startswith("-----"):
                    in_std = False; continue
                t = line.split()
                sym = ATOMIC_SYMBOLS.get(int(t[1]), f"Z{t[1]}")
                result["atoms"].append({"symbol": sym,
                    "x": float(t[3]), "y": float(t[4]), "z": float(t[5])})

    if result["scf_energies"]:
        result["final_scf_energy"] = result["scf_energies"][-1]

    return result

data = parse_gaussian("water.log")
H2K  = 627.509

print("━" * 55)
print("  Gaussian Summary — water.log (B3LYP/6-31G*)")
print("━" * 55)
print(f"  Normal termination : {data['normal_termination']}  ({data['timestamp']})")
print(f"  SCF steps          : {len(data['scf_energies'])}")
print(f"  Final SCF energy   : {data['final_scf_energy']:.10f} Hartree")
print()
print("  Thermochemistry:")
for k, v in data["thermo"].items():
    print(f"    {k:<36}: {v:.6f} Hartree  ({v*H2K:.2f} kcal/mol)")
print()
print("  Geometry:")
for i, a in enumerate(data["atoms"], 1):
    print(f"    {i}  {a['symbol']}  {a['x']:>10.6f}  {a['y']:>10.6f}  {a['z']:>10.6f}")
print("━" * 55)


The complete parser wraps everything developed in Sections 2–7 into a single function that accepts a filename and returns a structured dictionary. Returning a dictionary — rather than a tuple of separate values — makes the function composable: you can pass the result directly to other functions, save it as JSON, or collect results from multiple files into a list and analyse them with pandas.

---
## Summary

You now have a complete toolkit for reading Gaussian output files.

| Task | Key technique |
|------|--------------|
| Find SCF energies | `"SCF Done:" in line` → `line.split()[4]` |
| Extract thermochemistry | `stripped.startswith(search_str)` → `split()[-1]` |
| Check job success | `"Normal termination" in line` |
| Parse geometry | Boolean flag + header-skip counter |
| Bond lengths | Euclidean distance formula with `math.sqrt` |
| Bond angles | Dot-product / arccos formula |
| Combine everything | One-pass parser returning a structured dict |

### Where to go next

- **Multiple files**: wrap `parse_gaussian` in a loop to compare energies across
  a reaction — e.g., reactant, transition state, and product.
- **Reaction energetics**: subtract `G_total` values to compute ΔG‡ and ΔG_rxn.
- **matplotlib**: plot SCF energy vs. optimisation step or draw a reaction coordinate diagram.
- **cclib**: a dedicated Python library for parsing many quantum-chemistry codes,
  built on the same string-parsing ideas you just practiced.
